# Deterministic Fraud Rules Engine

The model score, calibrated risk, Balanced expected-cost decision, and rule action are separate layers. These are **ILLUSTRATIVE OPERATIONAL RULES** built only from available inference signals; IEEE-CIS has no verified blacklist, compromised-device, sanctions, merchant, or company-policy feeds. No rule reads labels at inference. Thresholds are distribution quantiles fixed from TRAIN and temporal TRAIN OOF before external VALIDATION is accessed.

In [1]:
import gc
import json
import sys
import time
from dataclasses import asdict
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from fraud_detection.data import merge_train_data
from fraud_detection.decisioning.costs import scenario_profiles
from fraud_detection.decisioning.evaluation import evaluate_policy
from fraud_detection.decisioning.policy import decide
from fraud_detection.decisioning.rules import RulesEngine, apply_review_capacity, derive_thresholds
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.features.composite import CompositeIdentity, create_composite_keys
from fraud_detection.models import prepare_engineered_matrices
from fraud_detection.models.calibration import Calibrator, validate_oof
from fraud_detection.models.ensemble import weighted_score_blend

started = time.perf_counter()
profiles = scenario_profiles()
calibrator = joblib.load(ROOT / "models/probability_calibrator.joblib")
assert calibrator.method == "platt"

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## TRAIN OOF signals and fold-local composite frequency

Fold 2 is calibrated using Fold 1; Fold 3 is calibrated using Folds 1 and 2. The composite frequency for each OOF fold is mapped from its strictly earlier TRAIN reference rows, matching the existing feature definition without using future frequency counts. Labels are joined only for retrospective evaluation.

In [2]:
full_rows = 590540
train_end = int(full_rows * 0.70)
raw_columns = ["TransactionID", "TransactionAmt", "card1", "addr1", "P_emaildomain", "isFraud"]
train_raw = pd.read_csv(
    ROOT / "data/raw/train_transaction.csv", nrows=train_end, usecols=raw_columns
)
oof_all = pd.read_csv(ROOT / "reports/ensemble_oof_predictions.csv")
raw_oof = validate_oof(oof_all)
oof_all["raw_score"] = raw_oof
pieces = []
for fold_number in (2, 3):
    previous = oof_all.fold < fold_number
    current = oof_all.fold == fold_number
    piece = oof_all.loc[current].copy()
    piece["calibrated_probability"] = (
        Calibrator("platt")
        .fit(raw_oof[previous], oof_all.loc[previous, "y_true"])
        .transform(raw_oof[current])
    )
    pieces.append(piece)
oof_eval = pd.concat(pieces, ignore_index=True)
oof_eval = oof_eval.merge(train_raw, on="TransactionID", how="left", validate="one_to_one")
assert len(oof_eval) == 82676 and oof_eval.TransactionAmt.notna().all()
assert (oof_eval.y_true == oof_eval.isFraud).all()
composite_identity = (
    CompositeIdentity("card1_addr1_p_emaildomain", ("card1", "addr1", "P_emaildomain")),
)
train_keys = create_composite_keys(train_raw, composite_identity).iloc[:, 0]
oof_keys = create_composite_keys(oof_eval, composite_identity).iloc[:, 0]
frequency = np.zeros(len(oof_eval), dtype=float)
for fold_number in (2, 3):
    eval_positions = np.flatnonzero(oof_eval.fold.to_numpy() == fold_number)
    first_id = int(oof_eval.TransactionID.iloc[eval_positions[0]])
    reference_keys = train_keys.loc[train_raw.TransactionID < first_id]
    reference_frequency = reference_keys.value_counts(normalize=True)
    frequency[eval_positions] = (
        oof_keys.iloc[eval_positions].map(reference_frequency).fillna(0).to_numpy()
    )
oof_eval["composite_frequency"] = frequency
print(
    "TRAIN OOF analysis rows:",
    len(oof_eval),
    "rare/unseen reference share:",
    (frequency == 0).mean(),
)

C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\fraud-detection-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


TRAIN OOF analysis rows: 82676 rare/unseen reference share: 0.12714693502346508


## Freeze rules and thresholds

Three rules use distribution cutoffs only: R01 HARD BLOCK at TRAIN p99 calibrated risk and TRAIN p95 amount; R02 SOFT REVIEW for TRAIN p95 component disagreement and risk above TRAIN p75; R03 SOFT REVIEW for bottom-decile fold-local composite frequency, risk above TRAIN p75, and amount above TRAIN median. Soft rules only escalate an APPROVE. All matching reasons are retained. HARD BLOCK wins, then higher priority, then rule ID. The calibrated probability stays unchanged.

In [3]:
thresholds = derive_thresholds(train_raw.TransactionAmt, oof_eval)
engine = RulesEngine(thresholds)
freeze_path = ROOT / "reports/rules_frozen_config.json"
freeze_path.write_text(json.dumps(engine.configuration(), indent=2) + "\n", encoding="utf-8")
print(json.dumps(engine.configuration(), indent=2))
print("Frozen config persisted before external VALIDATION access")

{
  "thresholds": {
    "high_amount": 444.95,
    "extreme_amount": 1104.0,
    "very_high_risk": 0.8832053868208527,
    "elevated_risk": 0.028448885793415184,
    "high_disagreement": 0.1637102772387074,
    "rare_composite_frequency": 0.0,
    "nontrivial_amount": 68.95
  },
  "rules": [
    {
      "rule_id": "R01",
      "name": "Extreme risk and high amount",
      "description": "TRAIN p99 calibrated risk and TRAIN p95 transaction amount",
      "rule_type": "HARD",
      "priority": 100,
      "action": "BLOCK",
      "reason_code": "EXTREME_RISK_HIGH_AMOUNT",
      "predicate": "extreme_risk_amount"
    },
    {
      "rule_id": "R02",
      "name": "Component disagreement",
      "description": "TRAIN p95 component disagreement with risk above TRAIN p75",
      "rule_type": "SOFT",
      "priority": 60,
      "action": "REVIEW",
      "reason_code": "MODEL_DISAGREEMENT",
      "predicate": "disagreement_review"
    },
    {
      "rule_id": "R03",
      "name": "Rare composi

In [4]:
def input_and_base(frame):
    decision_input = frame.loc[
        :, ["TransactionID", "calibrated_probability", "TransactionAmt"]
    ].copy()
    base = decide(decision_input, profiles["Balanced"])
    signals = frame.loc[
        :,
        [
            "TransactionID",
            "calibrated_probability",
            "TransactionAmt",
            "lightgbm_score",
            "xgboost_score",
            "composite_frequency",
        ],
    ].copy()
    signals["base_decision"] = base.decision.to_numpy()
    return signals, base


def policy_view(base, action, overflowed=None):
    view = base.copy(deep=True)
    view["decision"] = np.asarray(action)
    approve = view.expected_cost_approve.to_numpy()
    review = view.expected_cost_review.to_numpy()
    block = view.expected_cost_block.to_numpy()
    view["selected_expected_cost"] = np.where(
        view.decision.eq("APPROVE"), approve, np.where(view.decision.eq("REVIEW"), review, block)
    )
    if overflowed is not None:
        view["review_overflowed"] = np.asarray(overflowed)
    return view


def analyze(frame, label_column):
    signals, base = input_and_base(frame)
    result = engine.evaluate(signals)
    assert np.array_equal(result.calibrated_probability, signals.calibrated_probability)
    final = policy_view(base, result.final_decision)
    before = evaluate_policy(base, frame[label_column], profiles["Balanced"])
    after = evaluate_policy(final, frame[label_column], profiles["Balanced"])
    metrics = pd.DataFrame([{"policy": "base", **before}, {"policy": "base_plus_rules", **after}])
    records = []
    for rule in engine.rules:
        mask = result.triggered_rule_ids.map(
            lambda ids, rule_id=rule.rule_id: rule_id in ids
        ).to_numpy()
        selected = frame.loc[mask]
        records.append(
            {
                "rule_id": rule.rule_id,
                "trigger_count": int(mask.sum()),
                "trigger_rate": float(mask.mean()),
                "fraud_prevalence": float(selected[label_column].mean()) if mask.any() else np.nan,
                "transaction_amount": float(selected.TransactionAmt.sum()),
                "fraudulent_amount": float(
                    selected.loc[selected[label_column].eq(1), "TransactionAmt"].sum()
                ),
                "base_decisions_overridden": int(result.loc[mask, "rule_override"].sum()),
                "resulting_approve": int((result.loc[mask, "final_decision"] == "APPROVE").sum()),
                "resulting_review": int((result.loc[mask, "final_decision"] == "REVIEW").sum()),
                "resulting_block": int((result.loc[mask, "final_decision"] == "BLOCK").sum()),
            }
        )
    contributions = pd.DataFrame(records)
    combinations = (
        result.triggered_rule_ids.value_counts().rename_axis("rule_ids").reset_index(name="count")
    )
    overrides = pd.crosstab(result.base_decision, result.final_decision)
    return signals, base, result, metrics, contributions, combinations, overrides


(
    train_signals,
    train_base,
    train_result,
    train_metrics,
    train_contributions,
    train_combinations,
    train_overrides,
) = analyze(oof_eval, "y_true")
display(train_metrics, train_contributions, train_combinations.head(10), train_overrides)
train_metrics.to_csv(ROOT / "reports/rules_train_oof_policy_metrics.csv", index=False)
train_contributions.to_csv(ROOT / "reports/rules_train_oof_contributions.csv", index=False)

,policy,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,...,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,base,82676,3358,79318,578094.482,0.696248,0.806320,801,1095,1462,...,0,67940,0.821762,0.846454,12683,0.153406,0.146095,2053,0.024832,0.007451
1,base_plus_rules,82676,3358,79318,578094.482,0.720310,0.811252,700,1196,1462,...,0,66279,0.801672,0.826786,14344,0.173497,0.165763,2053,0.024832,0.007451


,rule_id,trigger_count,trigger_rate,fraud_prevalence,transaction_amount,fraudulent_amount,base_decisions_overridden,resulting_approve,resulting_review,resulting_block
0,R01,49,0.000593,0.959184,36539.000,35560.000,0,0,0,49
1,R02,1311,0.015857,0.069413,54967.986,2611.248,1311,0,1311,0
2,R03,373,0.004512,0.026810,38316.216,953.000,373,0,373,0


,rule_ids,count
0,(),80966
1,"(R02,)",1288
2,"(R03,)",350
3,"(R01,)",49
4,"(R02, R03)",23


final_decision,APPROVE,BLOCK,REVIEW
base_decision,,,
APPROVE,66279,0,1661
BLOCK,0,2053,0
REVIEW,0,0,12683


## Review capacity interaction

Order: base expected-cost action, rules, collect every REVIEW candidate, then rank by the existing economic priority and TransactionID. The 5% cap below is a fixed sensitivity setting from the earlier business phase. Overflow falls to the cheaper non-review action. HARD BLOCK is never downgraded. The primary base-versus-rules comparison above has no cap, matching frozen Balanced decisioning.

In [5]:
CAPACITY_RATE = 0.05
train_capped = apply_review_capacity(train_result, train_base, CAPACITY_RATE)
assert not (
    (train_result.pre_capacity_decision == "BLOCK") & (train_capped.final_decision != "BLOCK")
).any()
capacity_summary = {
    "capacity_rate": CAPACITY_RATE,
    "uncapped_review_count": int(train_result.final_decision.eq("REVIEW").sum()),
    "capped_review_count": int(train_capped.final_decision.eq("REVIEW").sum()),
    "overflow_count": int(train_capped.review_capacity_overflowed.sum()),
    "hard_blocks_preserved": True,
}
print(capacity_summary)

{'capacity_rate': 0.05, 'uncapped_review_count': 14344, 'capped_review_count': 4133, 'overflow_count': 10211, 'hard_blocks_preserved': True}


## One external VALIDATION evaluation

The persisted TRAIN configuration is now frozen. Rebuild the unchanged 467-feature champion and apply the existing Platt calibrator. VALIDATION labels enter only retrospective metrics. HOLDOUT is never loaded or assigned.

In [6]:
full_rows = 590540
train_end = int(full_rows * 0.70)
validation_end = train_end + int(full_rows * 0.15)
transaction_path = ROOT / "data/raw/train_transaction.csv"
sample = pd.read_csv(transaction_path, nrows=1000, low_memory=False)
dtypes = {
    name: ("float64" if pd.api.types.is_numeric_dtype(dtype) else "object")
    for name, dtype in sample.dtypes.items()
}
dtypes["TransactionID"] = "int64"
dtypes["TransactionDT"] = "int64"
dtypes["card1"] = "int64"
dtypes["isFraud"] = "int8"
del sample
transaction = pd.read_csv(transaction_path, nrows=validation_end, dtype=dtypes)
validation_max_id = int(transaction.TransactionID.iloc[-1])
identity_path = ROOT / "data/raw/train_identity.csv"
identity_rows_through_validation = 125565  # Fixed IEEE-CIS chronological split boundary.
identity = pd.read_csv(identity_path, nrows=identity_rows_through_validation, low_memory=False)
assert int(identity.TransactionID.iloc[-1]) <= validation_max_id
merged = merge_train_data(transaction, identity)
train = merged.iloc[:train_end]
validation = merged.iloc[train_end:validation_end]
del transaction, identity, merged
gc.collect()
prepared = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    scaling_batch_rows=len(train),
)
positions = {name: i for i, name in enumerate(prepared.feature_names)}
baseline_names = tuple(
    name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
selected_names = baseline_names + composite_te_features
assert len(selected_names) == 467 and len(set(selected_names)) == 467
assert "isFraud" not in selected_names and "y_true" not in selected_names
indices = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
x_train, x_validation = prepared.x_train[:, indices], prepared.x_evaluation[:, indices]
y_train = train.isFraud.astype("int8")
y_validation = validation.isFraud.astype("int8")
matrix_mib = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
lgbm = lgb.LGBMClassifier(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm.fit(x_train, y_train)
lgbm_score = lgbm.predict_proba(x_validation)[:, 1]
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
xgboost = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgboost.fit(x_train, y_train)
xgboost_score = xgboost.predict_proba(x_validation)[:, 1]
raw_validation = weighted_score_blend(
    lgbm_score, xgboost_score, lightgbm_weight=0.8, xgboost_weight=0.2
)
calibrated_validation = calibrator.transform(raw_validation)
assert abs(average_precision_score(y_validation, raw_validation) - 0.615265) < 0.000002
assert abs(roc_auc_score(y_validation, raw_validation) - 0.916224) < 0.000002

assert len(selected_names) == 467

In [7]:
validation_rules_frame = pd.DataFrame(
    {
        "TransactionID": validation.TransactionID.to_numpy(),
        "TransactionAmt": validation.TransactionAmt.to_numpy(),
        "isFraud": y_validation.to_numpy(),
        "calibrated_probability": calibrated_validation,
        "lightgbm_score": lgbm_score,
        "xgboost_score": xgboost_score,
    }
)
validation_identity = create_composite_keys(
    validation_rules_frame.join(
        validation[["card1", "addr1", "P_emaildomain"]].reset_index(drop=True)
    ),
    composite_identity,
).iloc[:, 0]
train_reference_frequency = train_keys.value_counts(normalize=True)
validation_rules_frame["composite_frequency"] = (
    validation_identity.map(train_reference_frequency).fillna(0).to_numpy()
)
assert json.loads(freeze_path.read_text(encoding="utf-8")) == engine.configuration()
(
    validation_signals,
    validation_base,
    validation_result,
    validation_metrics,
    validation_contributions,
    validation_combinations,
    validation_overrides,
) = analyze(validation_rules_frame, "isFraud")
display(
    validation_metrics,
    validation_contributions,
    validation_combinations.head(10),
    validation_overrides,
)
validation_metrics.to_csv(ROOT / "reports/rules_validation_policy_metrics.csv", index=False)
validation_contributions.to_csv(ROOT / "reports/rules_validation_contributions.csv", index=False)

,policy,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,...,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,base,88581,3042,85539,496907.588,0.690138,0.817614,709,1168,1165,...,0,73908,0.834355,0.855738,13033,0.147131,0.138709,1640,0.018514,0.005553
1,base_plus_rules,88581,3042,85539,496907.588,0.724326,0.825232,579,1298,1165,...,0,71200,0.803784,0.825600,15741,0.177702,0.168847,1640,0.018514,0.005553


,rule_id,trigger_count,trigger_rate,fraud_prevalence,transaction_amount,fraudulent_amount,base_decisions_overridden,resulting_approve,resulting_review,resulting_block
0,R01,29,0.000327,0.965517,27105.000,26474.000,0,0,0,29
1,R02,2259,0.025502,0.054449,107425.187,4097.329,2259,0,2259,0
2,R03,490,0.005532,0.014286,50326.403,634.389,490,0,490,0


,rule_ids,count
0,(),85844
1,"(R02,)",2218
2,"(R03,)",449
3,"(R02, R03)",41
4,"(R01,)",29


final_decision,APPROVE,BLOCK,REVIEW
base_decision,,,
APPROVE,71200,0,2708
BLOCK,0,1640,0
REVIEW,0,0,13033


## Figures, MLflow, report, and system lock

This final section records the frozen comparison, the rule configuration, and the full pre-HOLDOUT manifest.


In [8]:
import hashlib
import importlib.metadata

fig_dir = ROOT / "reports/figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(engine.rules))
ax.bar(x - 0.18, train_contributions.trigger_rate, 0.36, label="TRAIN OOF")
ax.bar(x + 0.18, validation_contributions.trigger_rate, 0.36, label="VALIDATION")
ax.set_xticks(x, train_contributions.rule_id)
ax.set(xlabel="Illustrative rule", ylabel="Trigger rate", title="TRAIN-frozen rule triggers")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "rule_trigger_rates.png", dpi=170)
plt.close(fig)

rows = pd.concat([train_metrics, validation_metrics], ignore_index=True)
labels = ["TRAIN base", "TRAIN + rules", "VAL base", "VAL + rules"]
fig, ax = plt.subplots(figsize=(8, 4.5))
bottom = np.zeros(len(rows))
for action, color in (("approve", "#7ba58c"), ("review", "#e7aa57"), ("block", "#ba6870")):
    values = rows[f"{action}_rate"].to_numpy()
    ax.bar(labels, values, bottom=bottom, label=action.upper(), color=color)
    bottom += values
ax.set(ylabel="Share of transactions", title="Frozen Balanced decisions before and after rules")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "base_vs_rules_decision_rates.png", dpi=170)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for i, metric in enumerate(("fraud_capture_rate", "fraud_amount_capture_rate")):
    axes[0].bar(np.arange(4) + (i - 0.5) * 0.35, rows[metric], width=0.35, label=metric)
axes[0].set_xticks(range(4), labels, rotation=15)
axes[0].set(ylabel="Modeled capture rate", title="Fraud capture")
axes[0].legend(fontsize=8)
for i, metric in enumerate(("legitimate_review_rate", "legitimate_block_rate")):
    axes[1].bar(np.arange(4) + (i - 0.5) * 0.35, rows[metric], width=0.35, label=metric)
axes[1].set_xticks(range(4), labels, rotation=15)
axes[1].set(ylabel="Share of legitimate transactions", title="Customer intervention")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / "rule_fraud_capture_customer_impact.png", dpi=170)
plt.close(fig)

tracking_uri = f"sqlite:///{(ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-rules-engine")
with mlflow.start_run(run_name="frozen-balanced-illustrative-rules") as run:
    mlflow.set_tags(
        {
            "stage": "rules-engine",
            "dataset": "ieee-cis",
            "model": "frozen-calibrated-ensemble",
            "rules": "illustrative",
        }
    )
    mlflow.log_params(
        {
            "rule_count": len(engine.rules),
            "primary_scenario": "Balanced",
            "review_capacity_primary": "uncapped",
            "review_capacity_sensitivity": CAPACITY_RATE,
            **{f"threshold_{key}": value for key, value in asdict(thresholds).items()},
        }
    )
    for split_name, table, contributions, result in (
        ("train_oof", train_metrics, train_contributions, train_result),
        ("validation", validation_metrics, validation_contributions, validation_result),
    ):
        for _, row in table.iterrows():
            prefix = f"{split_name}_{row.policy}"
            keys = (
                "approve_rate",
                "review_rate",
                "block_rate",
                "fraud_capture_rate",
                "fraud_amount_capture_rate",
                "legitimate_review_rate",
                "legitimate_block_rate",
                "modeled_total_cost",
                "fraudulent_amount_approved",
            )
            mlflow.log_metrics({f"{prefix}_{key}": float(row[key]) for key in keys})
        mlflow.log_metric(f"{split_name}_override_rate", float(result.rule_override.mean()))
        mlflow.log_metric(
            f"{split_name}_total_trigger_rate", float(result.triggered_rule_ids.map(bool).mean())
        )
        for _, row in contributions.iterrows():
            mlflow.log_metric(f"{split_name}_{row.rule_id}_trigger_rate", float(row.trigger_rate))
    mlflow.log_artifact(str(freeze_path))
    for name in (
        "rule_trigger_rates",
        "base_vs_rules_decision_rates",
        "rule_fraud_capture_customer_impact",
    ):
        mlflow.log_artifact(str(fig_dir / f"{name}.png"), artifact_path="figures")
    run_id = run.info.run_id
print("MLflow run:", run_id)


def markdown_table(table, columns):
    output = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for row in table.loc[:, columns].itertuples(index=False, name=None):
        output.append(
            "| "
            + " | ".join(
                f"{value:.6f}" if isinstance(value, float) else str(value) for value in row
            )
            + " |"
        )
    return "\n".join(output)


def overlap_text(combinations):
    counts = combinations.copy()
    counts["n_rules"] = counts.rule_ids.map(len)
    one = int(counts.loc[counts.n_rules == 1, "count"].sum())
    many = int(counts.loc[counts.n_rules > 1, "count"].sum())
    common = ", ".join(
        f"{ids}: {count}"
        for ids, count in counts.loc[counts.n_rules > 0, ["rule_ids", "count"]]
        .head(8)
        .itertuples(index=False, name=None)
    )
    return f"exactly one={one:,}; multiple={many:,}. Common combinations: {common or 'none'}."


metric_columns = [
    "policy",
    "approve_count",
    "review_count",
    "block_count",
    "approve_rate",
    "review_rate",
    "block_rate",
    "fraud_capture_rate",
    "fraud_amount_capture_rate",
    "fraudulent_amount_approved",
    "legitimate_review_rate",
    "legitimate_block_rate",
    "modeled_total_cost",
]
contrib_columns = list(train_contributions.columns)
train_trigger = int(train_result.triggered_rule_ids.map(bool).sum())
val_trigger = int(validation_result.triggered_rule_ids.map(bool).sum())
train_changed = int(train_result.rule_override.sum())
val_changed = int(validation_result.rule_override.sum())
train_ar = int(
    ((train_result.base_decision == "APPROVE") & (train_result.final_decision == "REVIEW")).sum()
)
train_block = int(
    ((train_result.base_decision != "BLOCK") & (train_result.final_decision == "BLOCK")).sum()
)
val_ar = int(
    (
        (validation_result.base_decision == "APPROVE")
        & (validation_result.final_decision == "REVIEW")
    ).sum()
)
val_block = int(
    (
        (validation_result.base_decision != "BLOCK") & (validation_result.final_decision == "BLOCK")
    ).sum()
)
rule_lines = "\n".join(
    f"- {rule.rule_id} **{rule.rule_type} {rule.action}**, priority {rule.priority}, "
    f"reason `{rule.reason_code}`: {rule.description}."
    for rule in engine.rules
)
threshold_lines = "\n".join(f"- {key}: {value:.9f}" for key, value in asdict(thresholds).items())
train_overrides_table = train_overrides.reset_index()
validation_overrides_table = validation_overrides.reset_index()
report = f"""# Deterministic Fraud Rules Engine

**ILLUSTRATIVE OPERATIONAL RULES.** IEEE-CIS supplies no confirmed blacklist, compromised-device, merchant, sanctions, or company-rule feeds. Model risk, Balanced expected-cost base action, and deterministic rule action are separate layers. Rules never change the score.

## Frozen upstream system and TRAIN-only development

467 features, 80% LightGBM / 20% XGBoost, frozen Platt calibrator, and frozen Balanced cost scenario. TRAIN OOF analysis uses 82,676 chronological Fold 2/3 predictions, each calibrated from earlier folds. Composite frequency comes from each fold's strictly earlier TRAIN reference. Thresholds were saved to `reports/rules_frozen_config.json` before external VALIDATION access. Fraud labels, VALIDATION results, and SHAP values did not select rules or cutoffs.

## Exact illustrative rules and TRAIN-derived thresholds

{rule_lines}

R01 uses calibrated p99 and amount p95. R02 uses disagreement p95 and calibrated p75. R03 uses fold-local composite frequency p10, calibrated p75, and amount median. All comparisons are inclusive; unseen identity frequency is zero. TRAIN p99 amount is context only, not an active cutoff.

{threshold_lines}

HARD BLOCK > SOFT REVIEW > base. Within a type, higher priority wins, then rule ID. All triggered rule IDs, names, and reasons are retained. Soft rules only escalate APPROVE. No rule lowers BLOCK or changes calibrated probability.

## TRAIN OOF rule behavior

Triggered {train_trigger:,}/{len(train_result):,} ({train_trigger / len(train_result):.4%}); overrides {train_changed:,}; APPROVE -> REVIEW {train_ar:,}; APPROVE/REVIEW -> BLOCK {train_block:,}.

{markdown_table(train_contributions, contrib_columns)}

Overlap: {overlap_text(train_combinations)}

Base-to-final decisions:

{markdown_table(train_overrides_table, list(train_overrides_table.columns))}

### Frozen Balanced base versus Balanced plus rules

{markdown_table(train_metrics, metric_columns)}

Capture assumes the existing illustrative 0.8 REVIEW fraud-detection rate. Modeled cost uses the same Balanced expected-cost formula. Its difference is not a measured saving.

## Review capacity

Order: base expected-cost decision, rules, collect REVIEW candidates, then existing economic benefit ranking and TransactionID tie break. Primary comparison is uncapped to match earlier Balanced analysis. Fixed 5% sensitivity: uncapped REVIEW {capacity_summary["uncapped_review_count"]:,}, capped REVIEW {capacity_summary["capped_review_count"]:,}, overflow {capacity_summary["overflow_count"]:,}. Overflow becomes the cheaper non-review action. HARD BLOCK remains BLOCK.

## External VALIDATION: one application of frozen rules

The champion reproduced PR-AUC {average_precision_score(y_validation, raw_validation):.6f} and ROC-AUC {roc_auc_score(y_validation, raw_validation):.6f}. Persisted thresholds were checked unchanged. Triggered {val_trigger:,}/{len(validation_result):,} ({val_trigger / len(validation_result):.4%}); overrides {val_changed:,}; APPROVE -> REVIEW {val_ar:,}; APPROVE/REVIEW -> BLOCK {val_block:,}. Rules were not revised after these outcomes.

{markdown_table(validation_contributions, contrib_columns)}

Overlap: {overlap_text(validation_combinations)}

Base-to-final decisions:

{markdown_table(validation_overrides_table, list(validation_overrides_table.columns))}

{markdown_table(validation_metrics, metric_columns)}

Fraud transaction/amount capture, fraudulent amount approved, legitimate REVIEW/BLOCK rates, and modeled cost appear above. Incremental customer friction is the difference in legitimate intervention rates. Triggered-row fraud prevalence is retrospective only. Figures are under `reports/figures/`.

## Limits and extension

These rules illustrate architecture, not real company policy. Disagreement signals uncertainty, not confirmed fraud. REVIEW effectiveness and costs are assumptions. Genuine blacklist, device, merchant, or regulatory predicates can be registered without changing model code when verified feeds and ownership exist.

## Runtime, memory, and MLflow

Notebook runtime: {time.perf_counter() - started:.1f} seconds. Prepared full TRAIN+VALIDATION feature matrices: {matrix_mib:.1f} MiB. MLflow experiment `fraud-detection-rules-engine`; run `{run_id}`; tracking URI `{tracking_uri}`. Thresholds, trigger/override rates, decision rates, capture, legitimate impact, modeled costs, config, and three figures are logged.

No model, feature, ensemble weight, calibrator, cost scenario, or risk score changed. HOLDOUT was not loaded, transformed, predicted, or evaluated.
"""
(ROOT / "reports/rules_engine_summary.md").write_text(report, encoding="utf-8")

versions = {
    name: importlib.metadata.version(name)
    for name in ("numpy", "pandas", "scikit-learn", "lightgbm", "xgboost", "mlflow")
}
config_hash = hashlib.sha256(freeze_path.read_bytes()).hexdigest()
calibrator_hash = hashlib.sha256(
    (ROOT / "models/probability_calibrator.joblib").read_bytes()
).hexdigest()
manifest = f"""# Frozen system manifest — pre-HOLDOUT lock

This is the exact system for any later HOLDOUT evaluation. This phase has not accessed HOLDOUT.

## Features and preprocessing

- Count: **467**, selected in `notebooks/17_rules_engine.ipynb` from `prepare_engineered_matrices` using the established feature-name selection.
- Retained families: raw transaction/identity, base frequency encodings, five historical amount features, 20 composite identity frequency/history features, and four composite temporal target encodings. No graph features. No SHAP inputs.
- Preprocessing: fold-local frequency maps, strict past-only history, training-only tabular preprocessing and StandardScaler. TRAIN full-fit scaling batch size = len(TRAIN). Full TRAIN reference for validation composite frequency.
- Four target-encoding entities: {", ".join(COMPOSITE_TARGET_ENTITIES)}. Alpha 50; neutral prior 0.5; historical FIT labels only; same-timestamp labels excluded; frozen final-model label delay **0 days**. The separate label-delay study made no model selection.

## Models, blend, and calibration

- LightGBM `LGBMClassifier(learning_rate=0.1, n_estimators=400, num_leaves=63, class_weight='balanced', random_state=42, n_jobs=-1, verbosity=-1)`.
- XGBoost `XGBClassifier(learning_rate=0.1, max_depth=7, n_estimators=300, tree_method='hist', eval_metric='aucpr', random_state=42, n_jobs=-1, scale_pos_weight=TRAIN_negative/TRAIN_positive)`. Full TRAIN scale_pos_weight = {scale_pos_weight:.9f}.
- Probability blend: **0.8 LightGBM + 0.2 XGBoost**. External VALIDATION PR-AUC 0.615265; ROC-AUC 0.916224.
- Platt calibrator: `models/probability_calibrator.joblib`; SHA-256 `{calibrator_hash}`. TRAIN OOF selection in `reports/probability_calibration_summary.md`.
- Boosters are rebuilt with fixed parameters and the established feature pipeline, not serialized as model artifacts. Rebuild code is in this executed notebook and matches notebook 16.

## Business decision and illustrative rules

- Frozen Balanced scenario: {asdict(profiles["Balanced"])}. Expected-cost formulas and decision tie order APPROVE, REVIEW, BLOCK are in `src/fraud_detection/decisioning/costs.py` and `policy.py`.
- Frozen rules: {", ".join(rule.rule_id + " " + rule.name for rule in engine.rules)}. Predicates, priorities, actions, reasons, and thresholds are in `reports/rules_frozen_config.json`; SHA-256 `{config_hash}`. Implementation: `src/fraud_detection/decisioning/rules.py`.
- HARD BLOCK > SOFT REVIEW > base. All matches recorded. Calibrated score unchanged.
- Primary comparison: uncapped REVIEW. Fixed 5% capacity sensitivity: rules first, all REVIEW candidates ranked by existing economic benefit and TransactionID, overflow to cheaper non-review action, HARD BLOCK retained. No cap tuned on VALIDATION.

## Software and artifact paths

- Python {sys.version.split()[0]}; packages: {versions}.
- TRAIN OOF `reports/ensemble_oof_predictions.csv`; raw sources `data/raw/train_transaction.csv`, `data/raw/train_identity.csv` bounded to TRAIN/VALIDATION; calibrator `models/probability_calibrator.joblib`; rules `reports/rules_frozen_config.json`; executed notebook `notebooks/17_rules_engine.ipynb`.
- MLflow tracking URI `{tracking_uri}`, experiment `fraud-detection-rules-engine`, run `{run_id}`.

Future HOLDOUT evaluation must use this lock without revising features, models, calibration, costs, rules, thresholds, or capacity behavior.
"""
(ROOT / "reports/frozen_system_manifest.md").write_text(manifest, encoding="utf-8")
print("Reports and manifest written")

2026/10/11 17:39:53 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-rules-engine' does not exist. Creating a new experiment.


MLflow run: b1144bf4a3c442238987416521639a07


Reports and manifest written
